# Qwen2.5-VL Step-DPO Fine-Tuning (Kaggle T4)

This notebook trains an FP16 LoRA adapter on the audited Step-DPO pairs with TRL's multimodal `DPOTrainer`. It defaults to a three-step smoke run; set `DPO_SMOKE_ONLY=0` for full training after the smoke assertions pass.

In [ ]:
# === Cell 1: Install one compatible stack before importing ML libraries ===
import os
import subprocess
import sys

PACKAGES = [
    'transformers==4.57.6',
    'trl==0.29.1',
    'peft==0.18.1',
    'accelerate==1.12.0',
    'datasets==4.4.1',
    'qwen-vl-utils==0.0.14',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES], check=True)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle secrets.')
except Exception:
    print('HF_TOKEN not configured; the public base model remains accessible.')

In [ ]:
# === Cell 2: Resolve project data and validate the runtime ===
import json
from pathlib import Path
import random

import accelerate
import datasets
import peft
import torch
import transformers
import trl

if Path('/kaggle/working').exists() and not Path('experiments/001_500_reasoning/data/step_dpo_pairs.jsonl').exists():
    repo_dir = Path('/tmp/prm_project')
    repo_url = 'https://github.com/yahorlahunovich/prm_project.git'
    revision = os.environ.get('PRM_REPO_REVISION')
    if not revision:
        raise RuntimeError('Set PRM_REPO_REVISION to the tested commit SHA for reproducible Kaggle runs.')
    if not repo_dir.exists():
        subprocess.run(['git', 'clone', repo_url, str(repo_dir)], check=True)
    subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin', revision], check=True)
    subprocess.run(['git', '-C', str(repo_dir), 'checkout', '--detach', revision], check=True)
    os.chdir(repo_dir)

seed = 42
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU is required. Run this notebook on a Kaggle T4 session.')
capability = torch.cuda.get_device_capability(0)
if capability[0] < 7:
    raise RuntimeError(f'Unsupported GPU {torch.cuda.get_device_name(0)} with compute capability {capability}; request a T4.')

print({
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'trl': trl.__version__,
    'peft': peft.__version__,
    'accelerate': accelerate.__version__,
    'datasets': datasets.__version__,
    'gpu': torch.cuda.get_device_name(0),
    'gpu_count': torch.cuda.device_count(),
})

In [ ]:
# === Cell 3: Load and validate the audited multimodal preference data ===
from datasets import Dataset
from PIL import Image

images_dir = Path('data/CharXiv/images')
data_path = Path('experiments/001_500_reasoning/data/step_dpo_pairs.jsonl')
if not data_path.exists():
    raise FileNotFoundError(f'Missing preference data: {data_path}')
if not images_dir.exists() or not any(images_dir.iterdir()):
    subprocess.run([sys.executable, 'scripts/download_images.py'], check=True)

with data_path.open(encoding='utf-8') as handle:
    raw_data = [json.loads(line) for line in handle if line.strip()]
if not raw_data:
    raise ValueError('Step-DPO dataset is empty.')

hf_data = {'prompt': [], 'chosen': [], 'rejected': [], 'images': []}
for row_number, item in enumerate(raw_data, start=1):
    required = ('question_id', 'image_path', 'question', 'chosen', 'rejected')
    missing = [key for key in required if not str(item.get(key, '')).strip()]
    if missing:
        raise ValueError(f'Row {row_number} has empty required fields: {missing}')

    image_path = Path(item['image_path']).resolve()
    if not image_path.exists():
        raise FileNotFoundError(f'Row {row_number} image is missing: {image_path}')
    with Image.open(image_path) as source:
        image = source.convert('RGB').copy()

    prompt_text = (
        'Analyze this chart. Provide step-by-step reasoning and a final answer.\n'
        + item['question'].strip()
    )
    prompt = [{
        'role': 'user',
        'content': [
            {'type': 'image'},
            {'type': 'text', 'text': prompt_text},
        ],
    }]
    prefix = item.get('prefix', '')
    hf_data['prompt'].append(prompt)
    hf_data['chosen'].append([{'role': 'assistant', 'content': prefix + item['chosen'].strip()}])
    hf_data['rejected'].append([{'role': 'assistant', 'content': prefix + item['rejected'].strip()}])
    hf_data['images'].append([image])

dataset = Dataset.from_dict(hf_data).shuffle(seed=seed)
print(f'Validated {len(dataset)} Step-DPO pairs and all referenced images.')
print(dataset)

In [ ]:
# === Cell 4: Load one FP16 policy model and its multimodal processor ===
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

model_id = 'Qwen/Qwen2.5-VL-3B-Instruct'
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    dtype=torch.float16,
    attn_implementation='sdpa',
    device_map={'': 0},
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model.enable_input_require_grads()
if hasattr(model, 'visual'):
    model.visual.requires_grad_(False)

# Bound chart resolution so chosen/rejected vision activations fit one 16 GB T4.
processor = AutoProcessor.from_pretrained(
    model_id,
    min_pixels=128 * 28 * 28,
    max_pixels=256 * 28 * 28,
)
processor.tokenizer.padding_side = 'left'
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

print(f'Model loaded on {next(model.parameters()).device}; vision encoder frozen.')

In [ ]:
# === Cell 5: Configure FP16 LoRA and modern VLM DPO ===
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer
from trl.trainer.dpo_trainer import DataCollatorForVisionPreference


class VisionPreferenceCollatorWithReferenceLogps:
    """Preserve cached DPO reference scores omitted by TRL's VLM collator."""

    def __init__(self, processor):
        self.vision_collator = DataCollatorForVisionPreference(processor)

    def __call__(self, examples):
        batch = self.vision_collator(examples)
        for key in ('ref_chosen_logps', 'ref_rejected_logps'):
            if key in examples[0]:
                batch[key] = torch.tensor([example[key] for example in examples], dtype=torch.float32)
        return batch


smoke_only = os.environ.get('DPO_SMOKE_ONLY', '1') == '1'
train_dataset = dataset.select(range(min(4, len(dataset)))) if smoke_only else dataset
out_dir = Path('/kaggle/working/dpo_qwen_vl' if Path('/kaggle/working').exists() else './dpo_qwen_vl')
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    task_type='CAUSAL_LM',
)
training_args = DPOConfig(
    output_dir=str(out_dir),
    beta=0.1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_train_epochs=3,
    max_steps=3 if smoke_only else -1,
    max_length=None,
    precompute_ref_log_probs=True,
    logging_steps=1 if smoke_only else 5,
    save_strategy='no' if smoke_only else 'steps',
    save_steps=50,
    save_total_limit=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataset_num_proc=1,
    remove_unused_columns=False,
    report_to='none',
    fp16=True,
    bf16=False,
    seed=seed,
    data_seed=seed,
)
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    data_collator=VisionPreferenceCollatorWithReferenceLogps(processor),
    train_dataset=train_dataset,
    processing_class=processor,
    peft_config=peft_config,
)
print(f'DPOTrainer initialized (smoke_only={smoke_only}, samples={len(train_dataset)}).')

In [ ]:
# === Cell 6: Assert one real VLM-DPO batch, then train and save ===
batch = next(iter(trainer.get_train_dataloader()))
required_batch_keys = {'pixel_values', 'image_grid_thw', 'completion_mask'}
missing_keys = required_batch_keys.difference(batch)
if missing_keys:
    raise AssertionError(f'Multimodal DPO batch is missing keys: {sorted(missing_keys)}')
if batch['pixel_values'].ndim < 2 or batch['image_grid_thw'].shape[-1] != 3:
    raise AssertionError('Vision tensors have invalid shapes.')
if not batch['completion_mask'].bool().any():
    raise AssertionError('The DPO completion mask contains no trainable tokens.')

# The standalone check runs before Trainer has populated cached reference scores.
precompute_ref_logps = trainer.precompute_ref_logps
trainer.precompute_ref_logps = False
try:
    with torch.no_grad():
        smoke_loss = trainer.compute_loss(trainer.model, batch)
finally:
    trainer.precompute_ref_logps = precompute_ref_logps
if not torch.isfinite(smoke_loss):
    raise FloatingPointError(f'Non-finite pre-training DPO loss: {smoke_loss.item()}')
print({
    'pixel_values_shape': tuple(batch['pixel_values'].shape),
    'image_grid_thw_shape': tuple(batch['image_grid_thw'].shape),
    'completion_tokens': int(batch['completion_mask'].sum()),
    'initial_dpo_loss': float(smoke_loss),
})
del batch, smoke_loss
torch.cuda.empty_cache()

train_result = trainer.train()
save_name = 'qwen_vl_step_dpo_smoke_adapter' if smoke_only else 'qwen_vl_step_dpo_adapter'
save_path = Path('/kaggle/working') / save_name if Path('/kaggle/working').exists() else Path(save_name)
trainer.save_model(str(save_path))
processor.save_pretrained(str(save_path))
print({'adapter_path': str(save_path), 'train_metrics': train_result.metrics})

In [ ]:
# === Cell 7: Verify saved adapter artifacts ===
required_artifacts = {'adapter_config.json', 'adapter_model.safetensors'}
saved_files = {path.name for path in save_path.iterdir()}
missing_artifacts = required_artifacts.difference(saved_files)
if missing_artifacts:
    raise AssertionError(f'Saved adapter is incomplete: {sorted(missing_artifacts)}')
print(f'Validated adapter artifacts in {save_path}: {sorted(saved_files)}')